In [1]:

import os
import pandas as pd

BASE = "/kaggle/input/notebooks/arvindreddy99920"

pca = pd.read_csv(
    f"{BASE}/validate-pca/pca_results/validation_details.csv"
)

quant = pd.read_csv(
    f"{BASE}/quantized-compression-01/quantized_validation/quantized_validation_details.csv"
)

# Select K=8
pca8 = pca[pca["components"] == 8].copy()

def clean_region(name):
    return name.lower().replace("_", " ").replace("-", " ").strip()

pca8["region"] = pca8["region"].apply(clean_region)
quant["region"] = quant["region"].apply(clean_region)

# Overall comparison
overall = pd.DataFrame({
    "Metric": ["PSNR (dB)", "SSIM", "Compression Ratio"],
    "Standard PCA": [
        pca8["PSNR"].mean(),
        pca8["SSIM"].mean(),
        pca8["compression_ratio"].mean()
    ],
    "Quantized PCA": [
        quant["PSNR"].mean(),
        quant["SSIM"].mean(),
        quant["Compression_Ratio"].mean()
    ]
})

# Region comparison
pca_region = pca8.groupby("region").agg(
    PCA_PSNR=("PSNR", "mean"),
    PCA_SSIM=("SSIM", "mean"),
    PCA_CR=("compression_ratio", "mean")
).reset_index()

quant_region = quant.groupby("region").agg(
    Quant_PSNR=("PSNR", "mean"),
    Quant_SSIM=("SSIM", "mean"),
    Quant_CR=("Compression_Ratio", "mean")
).reset_index()

region_summary = pd.merge(
    pca_region, quant_region, on="region"
)

# Output
print("\n" + "="*65)
print("             PCA vs QUANTIZED PCA")
print("="*65)

print(f"\nValidation patches : {len(pca8)}")
print("PCA components     : 8")
print("Regions            : 3")

print("\n" + "-"*65)
print("OVERALL PERFORMANCE")
print("-"*65)
print(overall.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n" + "-"*65)
print("REGION-WISE PERFORMANCE")
print("-"*65)

for _, row in region_summary.iterrows():
    print(f"\nRegion: {row['region'].upper()}")
    print(f"  PSNR             : {row['PCA_PSNR']:.4f} | {row['Quant_PSNR']:.4f}")
    print(f"  SSIM             : {row['PCA_SSIM']:.6f} | {row['Quant_SSIM']:.6f}")
    print(f"  Compression Ratio: {row['PCA_CR']:.4f} | {row['Quant_CR']:.4f}")

# Save results
OUT = "/kaggle/working/final_comparison"
os.makedirs(OUT, exist_ok=True)

overall.to_csv(f"{OUT}/overall_comparison.csv", index=False)
region_summary.to_csv(f"{OUT}/region_comparison.csv", index=False)

print("\n" + "="*65)
print("Comparison results saved successfully.")
print("="*65)



             PCA vs QUANTIZED PCA

Validation patches : 165
PCA components     : 8
Regions            : 3

-----------------------------------------------------------------
OVERALL PERFORMANCE
-----------------------------------------------------------------
           Metric  Standard PCA  Quantized PCA
        PSNR (dB)     47.989356      47.989354
             SSIM      0.992622       0.992622
Compression Ratio      4.181184       8.362211

-----------------------------------------------------------------
REGION-WISE PERFORMANCE
-----------------------------------------------------------------

Region: GOBI
  PSNR             : 50.2171 | 50.2171
  SSIM             : 0.998024 | 0.998024
  Compression Ratio: 3.6444 | 7.2886

Region: MURRAY DARLING
  PSNR             : 48.5126 | 48.5126
  SSIM             : 0.995680 | 0.995680
  Compression Ratio: 3.6932 | 7.3863

Region: TOKYO BAY
  PSNR             : 45.5000 | 45.5000
  SSIM             : 0.985692 | 0.985692
  Compression Ratio: 4.9

In [2]:
# ============================================================
# FINAL PCA vs QUANTIZED PCA VISUALIZATION AND SUMMARY
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# 1. Overall metric comparison
metrics = ["PSNR (dB)", "SSIM", "Compression Ratio"]

pca_values = [
    pca8["PSNR"].mean(),
    pca8["SSIM"].mean(),
    pca8["compression_ratio"].mean()
]

quant_values = [
    quant["PSNR"].mean(),
    quant["SSIM"].mean(),
    quant["Compression_Ratio"].mean()
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=metrics
)

for i in range(3):
    fig.add_trace(
        go.Bar(
            x=["Standard PCA", "Quantized PCA"],
            y=[pca_values[i], quant_values[i]],
            name=metrics[i],
            showlegend=False,
            text=[f"{pca_values[i]:.4f}", f"{quant_values[i]:.4f}"],
            textposition="auto"
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title="Overall Compression Performance Comparison",
    height=450,
    template="plotly_white"
)

fig.show()


# 2. Region-wise comparisons
regions = region_summary["region"].str.upper()

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Region-wise PSNR",
        "Region-wise SSIM",
        "Region-wise Compression Ratio"
    ]
)

region_metrics = [
    ("PCA_PSNR", "Quant_PSNR", "PSNR (dB)"),
    ("PCA_SSIM", "Quant_SSIM", "SSIM"),
    ("PCA_CR", "Quant_CR", "Compression Ratio")
]

for i, (pca_col, quant_col, metric_name) in enumerate(region_metrics):
    fig.add_trace(
        go.Bar(
            x=regions,
            y=region_summary[pca_col],
            name="Standard PCA",
            legendgroup="PCA",
            showlegend=(i == 0)
        ),
        row=1, col=i+1
    )

    fig.add_trace(
        go.Bar(
            x=regions,
            y=region_summary[quant_col],
            name="Quantized PCA",
            legendgroup="Quantized",
            showlegend=(i == 0)
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title="Region-wise PCA vs Quantized PCA Comparison",
    barmode="group",
    height=500,
    template="plotly_white"
)

fig.show()


# 3. Final numerical summary
psnr_diff = abs(pca_values[0] - quant_values[0])
ssim_diff = abs(pca_values[1] - quant_values[1])
ratio_improvement = quant_values[2] / pca_values[2]

print("\n" + "="*60)
print("                 FINAL PROJECT SUMMARY")
print("="*60)

print("Validation patches:", len(pca8))
print("PCA components: 8")

print("\nStandard PCA:")
print(f"  PSNR: {pca_values[0]:.6f} dB")
print(f"  SSIM: {pca_values[1]:.6f}")
print(f"  Compression ratio: {pca_values[2]:.4f}:1")

print("\nQuantized PCA:")
print(f"  PSNR: {quant_values[0]:.6f} dB")
print(f"  SSIM: {quant_values[1]:.6f}")
print(f"  Compression ratio: {quant_values[2]:.4f}:1")

print("\nDifferences:")
print(f"  Absolute PSNR difference: {psnr_diff:.8f} dB")
print(f"  Absolute SSIM difference: {ssim_diff:.8f}")
print(f"  Reported compression ratio factor: {ratio_improvement:.2f}x")




                 FINAL PROJECT SUMMARY
Validation patches: 165
PCA components: 8

Standard PCA:
  PSNR: 47.989356 dB
  SSIM: 0.992622
  Compression ratio: 4.1812:1

Quantized PCA:
  PSNR: 47.989354 dB
  SSIM: 0.992622
  Compression ratio: 8.3622:1

Differences:
  Absolute PSNR difference: 0.00000202 dB
  Absolute SSIM difference: 0.00000000
  Reported compression ratio factor: 2.00x
